# Tree Lab

## Preprocessing


In [1]:
# imports and config
import json
import logging
import os
import re
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
from tqdm import tqdm

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

cwd = Path.cwd()
root = cwd if (cwd / 'src').exists() else cwd.parents[1]
if str(root) not in sys.path:
    sys.path.append(str(root))

from src.llm.llm import OpenAILLM
from src.preprocessing.reader import Reader
from src.tree_rag.tree import DocumentTree, TreeNode

load_dotenv()

debug = False
logging.basicConfig(
    level=logging.DEBUG if debug else logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger('tree_lab')

base_url = os.getenv('BASE_OPEN_AI_URL')
api_key = os.getenv('OPEN_AI_KEY')
client = OpenAI(api_key=api_key, base_url=base_url)
llm = OpenAILLM(client=client, model_name='gpt-5.4-mini')
reader = Reader(logger=logger)

raw_data_path = root / 'data' / 'raw_data'
pdf_path = sorted(raw_data_path.glob('*.pdf'))[0]
pdf_path

/Users/danilaandreev/Documents/HSE/degree/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/Users/danilaandreev/Documents/HSE/degree/data/raw_data/4293720391.pdf')

In [2]:
# read pdf
document = reader.read(str(pdf_path))
assert document is not None, f'Failed to read {pdf_path}'

len(document.pages), document.pages[0].number, document.pages[0].text[:500]

(46,
 1,
 'МИНИСТЕРСТВО СТРОИТЕЛЬСТВА И ЖИЛИЩНО-КОММУНАЛЬНОГО ХОЗЯЙСТВА РОССИЙСКОЙ ФЕДЕРАЦИИ С В О Д П Р А В И Л СП 129.13330.2019 НАРУЖНЫЕ СЕТИ И СООРУЖЕНИЯ ВОДОСНАБЖЕНИЯ И КАНАЛИЗАЦИИ Актуализированная редакция СНиП 3.05.04—85* Издание официальное Москва Стандартинформ 2020 сертификат соответств кабели')

In [3]:
from pprint import pprint
len(document.pages[6].text)

4075

In [4]:
# schemas and helpers
from collections import Counter

MAX_TEXT_CHARS = 5000
MIN_TEXT_CHARS = 500
MAX_LLM_SPLIT_CHARS = 15000
MAX_SPLIT_DEPTH = 10
MAX_TITLE_CHARS = 80
MAX_TITLE_WORDS = 10


class TOCEntry(BaseModel):
    level: int = Field(ge=1, le=6)
    title: str
    page_number: int = Field(ge=1)


class TOCExtraction(BaseModel):
    has_toc: bool
    toc_pages: list[int] = Field(default_factory=list)
    entries: list[TOCEntry] = Field(default_factory=list)


class TOCPageDetection(BaseModel):
    has_toc: bool
    entries: list[TOCEntry] = Field(default_factory=list)


class PageMatch(BaseModel):
    matched: bool
    page_number: Optional[int] = None


class SplitPart(BaseModel):
    title: str
    anchor_start: str


class SplitDecision(BaseModel):
    can_split: bool
    parts: list[SplitPart] = Field(default_factory=list)


class ChunkTitle(BaseModel):
    title: str


@dataclass
class SectionSpan:
    level: int
    title: str
    start_page: int
    end_page: int


def normalize_text(text: str) -> str:
    text = text.lower().replace('ё', 'е')
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()


def pages_to_text(doc, start_page: int, end_page: int, with_tags: bool = True, skip_pages: Optional[set[int]] = None) -> str:
    skip_pages = skip_pages or set()
    parts = []
    for page in doc.pages[start_page - 1:end_page]:
        if page.number in skip_pages:
            continue
        if with_tags:
            parts.append(f"<page_{page.number}>\n{page.text}\n</page_{page.number}>")
        else:
            parts.append(page.text)
    return "\n\n".join(parts)


def detect_toc_on_page(page_number: int, page_text: str) -> TOCPageDetection:
    prompt = f"""
Тебе дана одна страница PDF-документа.
Нужно понять, есть ли на ней фрагмент оглавления.

Если на странице есть оглавление, извлеки все видимые пункты.

Правила:
- Оглавление - это последовательный список разделов документа с номерами страниц.
- Приложения тоже являются частью оглавления.
- Любая строка вида "Приложение А ...", "Приложение Б ..." и так далее должна быть извлечена как отдельный пункт.
- Все приложения размечай как level=1.
- Строку "Библиография" тоже извлекай как отдельный пункт с level=1.
- Если title перенесён или склеен OCR, восстанови один цельный title.
- Если номер страницы прилип к тексту, всё равно выдели правильный page_number.
- Игнорируй колонтитулы, коды документа и мусор.
- Если на странице нет оглавления, верни has_toc=false и пустой entries.
- Не придумывай пункты, которых нет на странице.

Номер страницы документа: {page_number}

Текст страницы:
{page_text}
"""
    return llm.parse(prompt, TOCPageDetection, temperature=0)


def merge_toc_entries(page_entries: list[list[TOCEntry]]) -> list[TOCEntry]:
    raw_lists = []
    for index, entries in enumerate(page_entries, start=1):
        raw_lists.append({
            "chunk_index": index,
            "entries": [entry.model_dump() for entry in entries],
        })

    prompt = f"""
Тебе дано несколько подряд идущих списков оглавления, извлечённых с соседних страниц документа.
Объедини их в один итоговый список.

Правила:
- Сохрани исходный порядок пунктов.
- Удали только явные дубликаты и OCR-мусор.
- Приложения являются частью оглавления и должны остаться в итоговом списке.
- Все приложения должны иметь level=1.
- Строку "Библиография" тоже нужно сохранить как пункт оглавления с level=1.
- Если приложение или библиография были разбиты OCR на части, склей их в один title.
- Не добавляй новые пункты, которых нет во входных списках.

Списки для объединения:
{json.dumps(raw_lists, ensure_ascii=False, indent=2)}
"""
    result = llm.parse(prompt, TOCPageDetection, temperature=0)
    return result.entries


def recover_appendix_entries(toc_text: str) -> list[TOCEntry]:
    prompt = f"""
Тебе дан сырой текст оглавления документа.
Извлеки из него только приложения и библиографию.

Правила:
- Извлекай только строки, начинающиеся с "Приложение", и строку "Библиография".
- Приложения являются частью оглавления.
- Все приложения должны иметь level=1.
- Библиография тоже должна иметь level=1.
- Если строка перенесена или склеена OCR, восстанови один цельный title и page_number.
- Не возвращай основные разделы документа.
- Если приложений и библиографии нет, верни has_toc=false и пустой entries.

Текст оглавления:
{toc_text}
"""
    result = llm.parse(prompt, TOCPageDetection, temperature=0)
    return result.entries


def extract_toc(doc, max_pages: int = 20) -> TOCExtraction:
    toc_page_numbers: list[int] = []
    toc_chunks: list[list[TOCEntry]] = []
    toc_started = False

    for page in tqdm(doc.pages[:max_pages]):
        page_result = detect_toc_on_page(page.number, page.text)

        if page_result.has_toc and page_result.entries:
            toc_started = True
            toc_page_numbers.append(page.number)
            toc_chunks.append(page_result.entries)
            continue

        if toc_started:
            break

    if not toc_chunks:
        return TOCExtraction(has_toc=False, toc_pages=[], entries=[])

    merged_entries = merge_toc_entries(toc_chunks)
    toc_text = pages_to_text(doc, toc_page_numbers[0], toc_page_numbers[-1], with_tags=False)
    has_appendix_in_text = 'приложение' in toc_text.lower()
    has_appendix_in_entries = any(entry.title.lower().startswith('приложение') for entry in merged_entries)
    has_bibliography_in_text = 'библиография' in toc_text.lower()
    has_bibliography_in_entries = any(entry.title.lower().startswith('библиография') for entry in merged_entries)

    if (has_appendix_in_text and not has_appendix_in_entries) or (has_bibliography_in_text and not has_bibliography_in_entries):
        appendix_entries = recover_appendix_entries(toc_text)
        known_keys = {(entry.title.strip().lower(), entry.page_number) for entry in merged_entries}
        for entry in appendix_entries:
            key = (entry.title.strip().lower(), entry.page_number)
            if key not in known_keys:
                merged_entries.append(entry)
                known_keys.add(key)

    return TOCExtraction(has_toc=True, toc_pages=toc_page_numbers, entries=merged_entries)


def find_title_page_global(title: str, doc, skip_pages: Optional[set[int]] = None, max_search_pages: int = 40) -> Optional[int]:
    skip_pages = skip_pages or set()
    normalized_title = normalize_text(title)
    if not normalized_title:
        return None

    for page in doc.pages[:max_search_pages]:
        if page.number in skip_pages:
            continue
        if normalized_title in normalize_text(page.text):
            return page.number
    return None


def estimate_page_offset(entries: list[TOCEntry], doc, toc_pages: list[int]) -> int:
    skip_pages = set(toc_pages)
    offsets = []
    top_entries = [entry for entry in entries if entry.level == 1][:6]
    for entry in top_entries:
        physical_page = find_title_page_global(entry.title, doc, skip_pages=skip_pages)
        if physical_page is not None:
            offsets.append(physical_page - entry.page_number)
    if not offsets:
        return 0
    return Counter(offsets).most_common(1)[0][0]


def find_section_start_page(title: str, doc, expected_page: int, window: int = 4, skip_pages: Optional[set[int]] = None) -> Optional[int]:
    skip_pages = skip_pages or set()
    normalized_title = normalize_text(title)
    if not normalized_title:
        return None

    start = max(1, expected_page - window)
    end = min(len(doc.pages), expected_page + window)
    for page_number in range(start, end + 1):
        if page_number in skip_pages:
            continue
        page_text = normalize_text(doc.pages[page_number - 1].text)
        if normalized_title in page_text:
            return page_number

    local_pages = pages_to_text(doc, start, end, with_tags=True, skip_pages=skip_pages)
    prompt = f"""
Тебе дано название раздела и несколько страниц документа.
Найди страницу, на которой начинается этот раздел.

Правила:
- Используй нестрогое сопоставление.
- Игнорируй разрывы пробелов и OCR-шум.
- Если по этим страницам нельзя уверенно понять, где начинается раздел, верни matched=false.

Название раздела: {title}

Страницы:
{local_pages}
"""
    result = llm.parse(prompt, PageMatch, temperature=0)
    return result.page_number if result.matched else None


def resolve_section_spans(entries: list[TOCEntry], doc, toc_pages: list[int]) -> list[SectionSpan]:
    resolved = []
    skip_pages = set(toc_pages)
    page_offset = estimate_page_offset(entries, doc, toc_pages)

    for entry in entries:
        adjusted_page = max(1, min(len(doc.pages), entry.page_number + page_offset))
        matched_page = find_section_start_page(entry.title, doc, adjusted_page, skip_pages=skip_pages)
        start_page = matched_page or adjusted_page
        resolved.append(SectionSpan(level=entry.level, title=entry.title, start_page=start_page, end_page=start_page))

    for index, item in enumerate(resolved):
        next_start = None
        for candidate in resolved[index + 1:]:
            if candidate.level <= item.level:
                next_start = candidate.start_page
                break
        item.end_page = (next_start - 1) if next_start else len(doc.pages)
        item.end_page = max(item.start_page, item.end_page)

    return resolved


def find_anchor_position(text: str, anchor: str, start_pos: int = 0) -> Optional[int]:
    anchor = anchor.strip()
    if not anchor:
        return None
    exact_pos = text.find(anchor, start_pos)
    if exact_pos != -1:
        return exact_pos
    tokens = anchor.split()
    if not tokens:
        return None
    pattern = r"\s+".join(re.escape(token) for token in tokens)
    match = re.search(pattern, text[start_pos:], flags=re.IGNORECASE)
    if match:
        return start_pos + match.start()
    return None


def split_text_by_pattern(text: str, pattern: re.Pattern[str]) -> list[str]:
    matches = list(pattern.finditer(text))
    if len(matches) < 2:
        return [text.strip()] if text.strip() else []
    positions = [match.start() for match in matches]
    segments = []
    for index, start in enumerate(positions):
        end = positions[index + 1] if index + 1 < len(positions) else len(text)
        segment = text[start:end].strip()
        if segment:
            segments.append(segment)
    return segments


def split_by_sentences(text: str) -> list[str]:
    parts = re.split(r"(?<=[.!?;:])\s+", text)
    return [part.strip() for part in parts if part.strip()]


def extract_units(text: str) -> list[str]:
    numbered_pattern = re.compile(r"(?=(?:^|(?<=\s))\d+(?:\.\d+)+\s+)")
    units = split_text_by_pattern(text, numbered_pattern)
    if len(units) > 1:
        return units
    blocks = [block.strip() for block in text.split("\n\n") if block.strip()]
    if len(blocks) > 1:
        return blocks
    sentences = split_by_sentences(text)
    if sentences:
        return sentences
    return [text.strip()] if text.strip() else []


def pack_units(units: list[str], max_chars: int, min_chars: int = MIN_TEXT_CHARS) -> list[str]:
    chunks: list[str] = []
    buffer = ""
    for unit in units:
        unit = unit.strip()
        if not unit:
            continue
        if len(unit) > max_chars:
            if buffer:
                chunks.append(buffer.strip())
                buffer = ""
            sub_units = split_by_sentences(unit)
            if len(sub_units) <= 1:
                sub_units = [unit[i:i + max_chars] for i in range(0, len(unit), max_chars)]
            chunks.extend(pack_units(sub_units, max_chars, min_chars))
            continue
        candidate = unit if not buffer else f"{buffer} {unit}"
        if len(candidate) <= max_chars:
            buffer = candidate
        else:
            if buffer:
                chunks.append(buffer.strip())
            buffer = unit
    if buffer:
        chunks.append(buffer.strip())
    if len(chunks) >= 2 and len(chunks[-1]) < min_chars:
        chunks[-2] = f"{chunks[-2]} {chunks[-1]}".strip()
        chunks.pop()
    return [chunk for chunk in chunks if chunk.strip()]


def force_split_text(text: str, max_chars: int) -> list[str]:
    units = extract_units(text)
    chunks = pack_units(units, max_chars)
    if not chunks:
        return [text[:max_chars].strip(), text[max_chars:].strip()] if len(text) > max_chars else [text.strip()]
    return chunks


def plan_text_split(title: str, text: str) -> SplitDecision:
    prompt = f"""
Тебе дан фрагмент документа.
Нужно решить, стоит ли разбивать его на несколько последовательных частей для дерева документа.

Цель:
- получить крупные, связные и удобные для поиска фрагменты текста;
- не дробить текст слишком мелко;
- сохранить исходный текст без переписывания.

Правила:
- Если текст можно оставить цельным, верни can_split=false.
- Если делишь текст, разбивай его только на крупные смысловые блоки.
- Не создавай отдельную часть для каждого короткого нумерованного пункта.
- Несколько соседних пунктов нужно объединять, если они относятся к одной теме.
- Каждая часть должна быть непрерывным фрагментом исходного текста.
- Если в тексте есть хорошие названия частей, используй их.
- Если явных названий нет, придумай короткий осмысленный title самостоятельно.
- Не переписывай текст частей.
- Для каждой части верни anchor_start - точный фрагмент начала этой части, скопированный из исходного текста.
- Не делай части короче {MIN_TEXT_CHARS} символов, если этого можно избежать.

Название текущего раздела: {title}

Исходный текст:
{text}
"""
    return llm.parse(prompt, SplitDecision, temperature=0)


def sanitize_title(title: str) -> str:
    title = re.sub(r"\s+", " ", title)
    title = title.strip()
    title = title.strip(" -:;,.")
    return title


def is_bad_title(title: str, text: str) -> bool:
    title = sanitize_title(title)
    if not title:
        return True
    if len(title) > MAX_TITLE_CHARS:
        return True
    if len(title.split()) > MAX_TITLE_WORDS:
        return True
    if re.match(r"^(сп|гост)\b", title.lower()):
        return True
    if re.match(r"^\d+(?:\.\d+)+", title):
        return True
    if re.match(r"^\d{3,}\.\d+", title):
        return True
    normalized_title = normalize_text(title)
    normalized_start = normalize_text(text[: min(len(text), 300)])
    if normalized_title and normalized_title in normalized_start and len(title) > 20:
        return True
    return False


def generate_chunk_title(parent_title: str, text: str, index: int) -> str:
    sample = text[: min(len(text), 3000)]
    prompt = f"""
Тебе дан фрагмент текста из раздела документа.
Дай для него короткий и понятный title на русском языке.

Правила:
- Верни только краткое название темы.
- Максимум 8 слов.
- Не копируй начало текста дословно.
- Не используй длинные номера пунктов и коды документа.
- Не пересказывай текст целиком.

Родительский раздел: {parent_title}

Текст:
{sample}
"""
    try:
        result = llm.parse(prompt, ChunkTitle, temperature=0)
        title = sanitize_title(result.title)
        if not is_bad_title(title, text):
            return title
    except Exception:
        pass
    return f"Часть {index}"


def try_semantic_split(title: str, text: str) -> Optional[list[tuple[str, str]]]:
    if len(text) > MAX_LLM_SPLIT_CHARS:
        return None
    decision = plan_text_split(title, text)
    if not decision.can_split or len(decision.parts) < 2:
        return None

    positions: list[tuple[int, str]] = []
    cursor = 0
    for part in decision.parts:
        pos = find_anchor_position(text, part.anchor_start, cursor)
        if pos is None:
            return None
        if positions and pos <= positions[-1][0]:
            return None
        positions.append((pos, sanitize_title(part.title)))
        cursor = pos + 1

    segments: list[tuple[str, str]] = []
    for index, (start_pos, part_title) in enumerate(positions, start=1):
        end_pos = positions[index][0] if index < len(positions) else len(text)
        part_text = text[start_pos:end_pos].strip()
        if len(part_text) < MIN_TEXT_CHARS:
            return None
        if is_bad_title(part_title, part_text):
            part_title = generate_chunk_title(title, part_text, index)
        segments.append((part_title, part_text))
    return segments


def build_text_children(title: str, text: str, pages: list[int], new_id, nodes: dict[int, TreeNode], depth: int = 0) -> list[int]:
    text = text.strip()
    if not text:
        return []
    if len(text) <= MAX_TEXT_CHARS:
        leaf_id = new_id()
        nodes[leaf_id] = TreeNode(
            title="text",
            node_id=leaf_id,
            pages=pages,
            text=text,
            text_type="section_text",
        )
        return [leaf_id]

    if depth >= MAX_SPLIT_DEPTH:
        forced_chunks = force_split_text(text, MAX_TEXT_CHARS)
        child_ids: list[int] = []
        for index, chunk_text in enumerate(forced_chunks, start=1):
            section_id = new_id()
            chunk_title = generate_chunk_title(title, chunk_text, index)
            nodes[section_id] = TreeNode(
                title=chunk_title,
                node_id=section_id,
                pages=pages,
                nodes=build_text_children(chunk_title, chunk_text, pages, new_id, nodes, depth + 1),
            )
            child_ids.append(section_id)
        return child_ids

    if len(text) > MAX_LLM_SPLIT_CHARS:
        coarse_chunks = force_split_text(text, MAX_LLM_SPLIT_CHARS)
        child_ids: list[int] = []
        for index, chunk_text in enumerate(coarse_chunks, start=1):
            chunk_title = generate_chunk_title(title, chunk_text, index)
            section_id = new_id()
            nodes[section_id] = TreeNode(
                title=chunk_title,
                node_id=section_id,
                pages=pages,
                nodes=build_text_children(chunk_title, chunk_text, pages, new_id, nodes, depth + 1),
            )
            child_ids.append(section_id)
        return child_ids

    semantic_parts = try_semantic_split(title, text)
    if semantic_parts:
        child_ids: list[int] = []
        for index, (part_title, part_text) in enumerate(semantic_parts, start=1):
            if is_bad_title(part_title, part_text):
                part_title = generate_chunk_title(title, part_text, index)
            section_id = new_id()
            nodes[section_id] = TreeNode(
                title=part_title,
                node_id=section_id,
                pages=pages,
                nodes=build_text_children(part_title, part_text, pages, new_id, nodes, depth + 1),
            )
            child_ids.append(section_id)
        return child_ids

    forced_chunks = force_split_text(text, MAX_TEXT_CHARS)
    child_ids: list[int] = []
    for index, chunk_text in enumerate(forced_chunks, start=1):
        chunk_title = generate_chunk_title(title, chunk_text, index)
        section_id = new_id()
        nodes[section_id] = TreeNode(
            title=chunk_title,
            node_id=section_id,
            pages=pages,
            nodes=build_text_children(chunk_title, chunk_text, pages, new_id, nodes, depth + 1),
        )
        child_ids.append(section_id)
    return child_ids


def validate_tree_text_limits(tree: DocumentTree, max_chars: int = MAX_TEXT_CHARS) -> None:
    oversized = [node for node in tree.nodes.values() if node.is_leaf and len(node.text or '') > max_chars]
    if oversized:
        raise ValueError(f"Found oversized leaf nodes: {[node.node_id for node in oversized]}")


def build_tree_from_spans(doc, spans: list[SectionSpan]) -> DocumentTree:
    nodes: dict[int, TreeNode] = {}
    next_id = 1

    def new_id() -> int:
        nonlocal next_id
        node_id = next_id
        next_id += 1
        return node_id

    root_id = new_id()
    root = TreeNode(
        title=Path(doc.source_path).stem,
        node_id=root_id,
        pages=[1, len(doc.pages)],
        nodes=[],
    )
    nodes[root_id] = root

    stack: list[tuple[int, int]] = [(0, root_id)]
    for span in spans:
        while stack and stack[-1][0] >= span.level:
            stack.pop()
        parent_id = stack[-1][1]
        section_id = new_id()
        pages = list(range(span.start_page, span.end_page + 1))
        text = pages_to_text(doc, span.start_page, span.end_page, with_tags=False)
        nodes[section_id] = TreeNode(
            title=span.title,
            node_id=section_id,
            pages=pages,
            nodes=build_text_children(span.title, text, pages, new_id, nodes, depth=0),
        )
        parent = nodes[parent_id]
        if parent.nodes is None:
            parent.nodes = []
        parent.nodes.append(section_id)
        stack.append((span.level, section_id))

    tree = DocumentTree(root_id=root_id, nodes=nodes)
    tree.collapse_single_leaf_children()
    validate_tree_text_limits(tree)
    return tree


In [5]:
# extract toc
toc = extract_toc(document)

print('has_toc =', toc.has_toc)
print('toc_pages =', toc.toc_pages)
print('entries =', len(toc.entries))
toc.entries

 15%|█▌        | 3/20 [00:11<01:06,  3.93s/it]
2026-04-28 22:56:38,559 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"


has_toc = True
toc_pages = [3]
entries = 17


[TOCEntry(level=1, title='Область применения', page_number=1),
 TOCEntry(level=1, title='Нормативные ссылки', page_number=1),
 TOCEntry(level=1, title='Термины и определения', page_number=2),
 TOCEntry(level=1, title='Общие положения', page_number=2),
 TOCEntry(level=1, title='Земляные работы', page_number=2),
 TOCEntry(level=1, title='Монтаж трубопроводов', page_number=2),
 TOCEntry(level=1, title='Переходы трубопроводов через естественные и искусственные преграды', page_number=18),
 TOCEntry(level=1, title='Сооружения водоснабжения и канализации', page_number=19),
 TOCEntry(level=1, title='Дополнительные требования к строительству трубопроводов и сооружений водоснабжения и канализации в особых природных и климатических условиях', page_number=21),
 TOCEntry(level=1, title='Испытание трубопроводов и сооружений', page_number=22),
 TOCEntry(level=1, title='Приложение А Порядок проведения промывки и дезинфекции трубопроводов и сооружений хозяйственно-питьевого водоснабжения', page_number=

In [6]:
# resolve page spans
assert toc.has_toc and toc.entries, 'TOC was not extracted'
spans = resolve_section_spans(toc.entries, document, toc.toc_pages)
spans

2026-04-28 22:56:42,247 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
2026-04-28 22:56:43,929 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
2026-04-28 22:56:45,832 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"


[SectionSpan(level=1, title='Область применения', start_page=5, end_page=5),
 SectionSpan(level=1, title='Нормативные ссылки', start_page=5, end_page=5),
 SectionSpan(level=1, title='Термины и определения', start_page=6, end_page=6),
 SectionSpan(level=1, title='Общие положения', start_page=6, end_page=6),
 SectionSpan(level=1, title='Земляные работы', start_page=6, end_page=6),
 SectionSpan(level=1, title='Монтаж трубопроводов', start_page=7, end_page=21),
 SectionSpan(level=1, title='Переходы трубопроводов через естественные и искусственные преграды', start_page=22, end_page=22),
 SectionSpan(level=1, title='Сооружения водоснабжения и канализации', start_page=23, end_page=24),
 SectionSpan(level=1, title='Дополнительные требования к строительству трубопроводов и сооружений водоснабжения и канализации в особых природных и климатических условиях', start_page=25, end_page=25),
 SectionSpan(level=1, title='Испытание трубопроводов и сооружений', start_page=26, end_page=36),
 SectionSpan(l

In [7]:
# build tree
tree = build_tree_from_spans(document, spans)
root_node = tree.get_node(tree.root_id)
print(root_node.preview(depth=3))

2026-04-28 22:56:50,439 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
2026-04-28 22:56:52,966 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
2026-04-28 22:56:55,353 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
2026-04-28 22:56:58,532 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
2026-04-28 22:56:59,964 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
2026-04-28 22:57:02,015 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
2026-04-28 22:57:03,135 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
2026-04-28 22:57:04,680 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
2026-04-28 22:57:08,260 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "

4293720391
├── Область применения [2618 chars]
├── Нормативные ссылки [2618 chars]
├── Термины и определения [3569 chars]
├── Общие положения [3569 chars]
├── Земляные работы [3569 chars]
├── Монтаж трубопроводов
│   ├── Требования к монтажу трубопроводов
│   │   ├── Общие требования к монтажу [4072 chars]
│   │   └── Сварка и сборка стальных труб
│   ├── Соединения стеклокомпозитных труб
│   │   ├── Требования и виды соединений [1162 chars]
│   │   ├── Муфтовые, блокирующие и уплотнительные соединения [2130 chars]
│   │   ├── Монтаж и стыковка муфтой
│   │   ├── Фланцевые и механические соединения [3398 chars]
│   │   └── Клеевые и ламинированные соединения [2603 chars]
│   └── Уплотнение стыков железобетонных труб [3918 chars]
├── Переходы трубопроводов через естественные и искусственные преграды [3237 chars]
├── Сооружения водоснабжения и канализации
│   ├── Требования к водозаборным сооружениям [4837 chars]
│   └── Монтаж и испытания водозаборных сооружений [3291 chars]
├── Дополни

In [15]:
first_section_id = tree.root.nodes[0]
first_section = tree.get_node(first_section_id)

first_section.title, first_section.pages[:5], first_section.text[:2000]

('Область применения',
 [5],
 'СП 129.13330.2019 С В О Д П Р А В И Л Н А Р У Ж Н Ы Е С Е Т И И С О О Р У Ж Е Н И Я В О Д О С Н А Б Ж Е Н И Я И К А Н А Л И З А Ц И И External water supply and sewage networks and structures Дата введения — 2020— 07— 01 1 Область применения Настоящий свод правил распространяется на наружные сети и сооружения водоснабжения и ка\xad нализации и устанавливает требования, которые должны соблюдаться при проектировании вновь стро\xad ящихся и реконструируемых наружных сетей и сооружений водоснабжения и канализации населенных пунктов и промышленных предприятий. 2 Нормативные ссылки В настоящем своде правил использованы нормативные ссылки на следующие документы: ГОСТ 2405—88 Манометры, вакуумметры, мановакуумметры, напоромеры, тягомеры и тягонапо- ромеры. Общие технические условия ГОСТ 6019—83 Счетчики холодной воды крыльчатые. Общие технические условия ГОСТ 6718—93 (ИСО 2120—72, ИСО 5173—72) Хлор жидкий. Технические условия ГОСТ 6996—66 (ИСО 4136—89, ИСО 5173—81

In [14]:
# save tree
out_dir = root / 'src' / 'tree_rag' / 'data' / 'docs_tree'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'{pdf_path.stem}_tree.json'
tree.save(out_path)
out_path

PosixPath('/Users/danilaandreev/Documents/HSE/degree/src/tree_rag/data/docs_tree/4293720391_tree.json')

In [ ]:
# TODO:
# - сделать парсинг или генерацию названия документа
# - перенести препроцессинг в отдельный файл
# - сделать нормальный preview дерева, который подходит для LLM контекста
# - сделать preview текста в конкретном узле
# - сделать генерацию pydantic схемы, чтобы агент мог выбирать какую главу он хочет открыть из оглавления

# - сделать препроцессинг текста в эмбеддинги, чтобы быстро находить зону интереса + выводить зону интереса в оглавлении
# - сделать простого поискового агента 